# AIoT 作業二 (HW2)：多元線性回歸 (Multiple Linear Regression) 完整分析
**學號**：7115056029  
**資料集**：Kaggle Playground Series s4e5 (Flood Prediction Dataset)  
**資料集來源與連結**：[https://www.kaggle.com/competitions/playground-series-s4e5](https://www.kaggle.com/competitions/playground-series-s4e5)  
**分析流程**：遵循 **CRISP-DM** 六大階段 (Business Understanding -> Data Understanding -> Data Preparation -> Modeling -> Evaluation -> Deployment)

---
### 作業規範核對：
- [x] **特徵數限制**：具備 20 個數值特徵 (符合 10~20 特徵規範，亦符 14~39 範圍)
- [x] **分析任務**：多元線性回歸 (OLS) + 嶺回歸 (Ridge) + 套索回歸 (Lasso)
- [x] **特徵選擇**：RFE (Top 14, 滿足 >13 限制)、SelectKBest (Top 10)
- [x] **模型評估**：R^2, Adj R^2, RMSE, MAE 與五大古典線性回歸假設檢定
- [x] **預測圖需求**：繪製真實值 vs 預測值圖，並清楚標註 **95% 信賴區間 (CI)** 與 **95% 預測區間 (PI)**
- [x] **系統部署**：以 Flask 開發互動式洪澇風險預警決策網站

## 1. Business Understanding (商業與領域理解)
洪澇災害對全球都市發展與民生安全構成巨額威脅。本專案以 **Kaggle Playground Series s4e5** 為基礎，針對 20 項水文、氣象、地質、工程防護及治理政策指標，建立多元線性回歸計量模型。

**主要商業決策價值：**
1. **早期預警 (Early Warning)**：根據即時季風強度與各指標估計洪澇發生機率。
2. **邊際效益評估**：利用可解釋之回歸係數，評估排水系統改善、河道清淤與大壩加固等工程投資之邊際防洪效益。
3. **區間風險管理**：提供 95% 信賴區間 (母體平均風險) 與 95% 預測區間 (單一事件最壞情境防範)。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False
print("環境與模組載入成功！")

## 2. Data Understanding (資料理解)
載入資料集，檢視 20 項輸入特徵與目標變數 FloodProbability 的維度、缺失值及敘述統計。

In [2]:
data_path = "data/flood_prediction_data.csv"
if not os.path.exists(data_path):
    from generate_data import generate_kaggle_s4e5_flood_data
    df = generate_kaggle_s4e5_flood_data()
else:
    df = pd.read_csv(data_path)

features = [c for c in df.columns if c != "FloodProbability"]
target = "FloodProbability"
print(f"資料維度: {df.shape[0]} 樣本, {len(features)} 特徵")
print(f"缺失值總數: {df.isnull().sum().sum()}")
df.describe().T[["mean", "std", "min", "50%", "max"]]

## 3. Data Preparation (資料準備)
檢定多重共線性 (Variance Inflation Factor, VIF)，並進行 80/20 訓練與測試集切分。

In [3]:
from numpy.linalg import inv
corr = np.corrcoef(df[features].values, rowvar=False)
vif_values = np.diag(inv(corr))
vif_df = pd.DataFrame({"Feature": features, "VIF": vif_values})
print(f"最大 VIF: {vif_df['VIF'].max():.4f} (均遠低於 5.0 門檻，無共線性危機)")

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"訓練集: {X_train.shape[0]} 筆, 測試集: {X_test.shape[0]} 筆")

## 4. Modeling & Feature Selection (建模與特徵選擇)
1. **OLS 全特徵模型** (20 特徵)
2. **RFE 特徵選擇** (挑選 Top 14 特徵，符合作業 13 < 特徵 < 40 規範)
3. **SelectKBest (F-regression)** (Top 10 特徵)
4. **Ridge & Lasso 正規化回歸**

In [4]:
# 4.1 OLS
ols = LinearRegression().fit(X_train, y_train)

# 4.2 RFE 挑選 14 個特徵
rfe = RFE(estimator=LinearRegression(), n_features_to_select=14, step=1).fit(X_train, y_train)
top_14 = [f for f, s in zip(features, rfe.support_) if s]
rfe_model = LinearRegression().fit(X_train[top_14], y_train)

# 4.3 SelectKBest 挑選 10 個特徵
kbest = SelectKBest(score_func=f_regression, k=10).fit(X_train, y_train)
top_10 = [f for f, s in zip(features, kbest.get_support()) if s]
kbest_model = LinearRegression().fit(X_train[top_10], y_train)

# 4.4 Ridge & Lasso
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
lasso = Lasso(alpha=0.001).fit(X_train, y_train)

print("Top 14 特徵 (RFE):", top_14)

## 5. Evaluation & Diagnostics (模型評估與五大假設檢定)
檢定回歸五大統計假設：
1. 線性性 (Linearity)
2. 殘差常態性 (Normality, Jarque-Bera)
3. 殘差同質變異性 (Homoscedasticity)
4. 誤差獨立性 (Independence, Durbin-Watson)
5. 無多重共線性 (Multicollinearity, VIF)

In [5]:
models_eval = {
    "OLS (All 20 Features)": (ols, X_train, X_test),
    "OLS (Top 14 - RFE)": (rfe_model, X_train[top_14], X_test[top_14]),
    "OLS (Top 10 - KBest)": (kbest_model, X_train[top_10], X_test[top_10]),
    "Ridge (alpha=1.0)": (ridge, X_train, X_test),
    "Lasso (alpha=0.001)": (lasso, X_train, X_test)
}

records = []
for name, (mod, xtr, xte) in models_eval.items():
    ypred = mod.predict(xte)
    r2 = r2_score(y_test, ypred)
    adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - xte.shape[1] - 1)
    rmse = np.sqrt(mean_squared_error(y_test, ypred))
    mae = mean_absolute_error(y_test, ypred)
    records.append({"Model": name, "Features": xte.shape[1], "Test R2": r2, "Adj R2": adj_r2, "RMSE": rmse, "MAE": mae})

eval_df = pd.DataFrame(records)
eval_df

### 預測圖：繪製真實值 vs 預測值 (含 95% 信賴區間 CI 與 95% 預測區間 PI)

In [6]:
y_pred_ols = ols.predict(X_test)
residuals = y_test.values - y_pred_ols
n, p = X_test.shape
dof = n - p - 1
residual_std = np.sqrt(np.sum(residuals**2) / dof)

sort_idx = np.argsort(y_pred_ols)
y_test_sorted = y_test.values[sort_idx]
y_pred_sorted = y_pred_ols[sort_idx]
X_test_sorted = X_test.values[sort_idx]

X_train_design = np.column_stack([np.ones(len(X_train)), X_train.values])
XtX_inv = np.linalg.pinv(X_train_design.T @ X_train_design)
X_test_design = np.column_stack([np.ones(len(X_test_sorted)), X_test_sorted])
leverage = np.sum((X_test_design @ XtX_inv) * X_test_design, axis=1)

t_crit = stats.t.ppf(1 - 0.05 / 2, df=dof)
se_mean = np.sqrt(residual_std**2 * leverage)
se_pred = np.sqrt(residual_std**2 * (1.0 + leverage))

ci_upper = y_pred_sorted + t_crit * se_mean
ci_lower = y_pred_sorted - t_crit * se_mean
pi_upper = y_pred_sorted + t_crit * se_pred
pi_lower = y_pred_sorted - t_crit * se_pred

plt.figure(figsize=(10, 6))
x_axis = np.arange(len(y_pred_sorted))
plt.fill_between(x_axis, pi_lower, pi_upper, color="#cbd5e1", alpha=0.55, label="95% Prediction Interval (PI)")
plt.fill_between(x_axis, ci_lower, ci_upper, color="#60a5fa", alpha=0.65, label="95% Confidence Interval (CI)")
plt.plot(x_axis, y_pred_sorted, color="#1d4ed8", linewidth=2.2, label="Fitted Regression Line")
plt.scatter(x_axis[::4], y_test_sorted[::4], color="#dc2626", alpha=0.45, s=15, label="Actual Test Data")
plt.title(f"Multiple Linear Regression: Predictions with 95% CI & PI (Test R2 = {eval_df.loc[0, 'Test R2']:.4f})", fontsize=12, fontweight="bold")
plt.xlabel("Ranked Test Observations", fontsize=10)
plt.ylabel("Flood Probability", fontsize=10)
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig("static/images/actual_vs_predicted.png", dpi=300)
print("預測圖繪製完成！")

## 6. Deployment (系統部署)
已將訓練完備之多元線性回歸模型導出為 Flask Web 應用。
執行 `python app.py` 即可於本機 `http://127.0.0.1:5000` 進行即時洪澇風險推論與情境模擬。